# 온도센서 실측 데이터 분석 - 전체 파이프라인

펠티어 6채널 제어값(4,096개 조합 x 3회 측정, 총 12,288행) 실측 데이터를
결측치 처리 -> 이상치 처리 -> 파생변수 생성 -> 채널별 영향력 확인 -> 
불량 판정 기준 수립 -> 머신러닝 모델링까지 전 과정을 진행한다.


## 1. 데이터 불러오기

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 윈도우 환경 한글 폰트 설정
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

FILE_PATH = r"C:\Users\hwan9\OneDrive\Desktop\프로젝트\스마트 팩토리\온도센서_csv\온도센서측정.csv"

df = pd.read_csv(FILE_PATH, encoding='utf-8-sig', dtype={'Peltier_Level(6digits)': str})

print("shape:", df.shape)
df.head(10)

In [ ]:
sensor_cols = ['Ref_Temp', 'Sec_2', 'Sec_3', 'Sec_4', 'Sec_5', 'Sec_6', 'Sec_7']

print("컬럼:", list(df.columns))
print("고유 제어값 조합 수:", df['Peltier_Level(6digits)'].nunique(), "/ 4096")
print("중복행 개수:", df.duplicated().sum())

## 2. 결측치 현황 파악

### 2-1. 컬럼(열)별 결측치 개수

In [ ]:
na_counts = df[sensor_cols].isnull().sum()
print(na_counts)
print()
print("합계:", na_counts.sum())

특정 센서에 몰리지 않고 7개 컬럼에 고르게 분산 -> 하드웨어 고장이 아닌 순간적 통신 오류로 판단.

### 2-2. 조합(제어값)별 결측 패턴 확인

In [ ]:
df['has_na'] = df[sensor_cols].isnull().any(axis=1)
print("결측치가 하나라도 포함된 행 수:", df['has_na'].sum())

group_na = df.groupby('Peltier_Level(6digits)')['has_na'].sum()
print()
print("조합별 결측 행 개수 분포 (0=3회 다 정상, 3=3회 다 결측):")
print(group_na.value_counts().sort_index())

## 3. 1단계 처리 - 같은 조합 3회가 전부 결측인 경우

참고할 실측값이 하나도 없어 되살릴 방법이 없음 -> **행 삭제**.

In [ ]:
all3_missing_codes = group_na[group_na == 3].index.tolist()
print(f"3회 전부 결측인 조합 개수: {len(all3_missing_codes)}개")

## 4. 2단계 처리 - 같은 조합 3회 중 2회가 결측(1회만 유효값)인 경우

남은 값이 1개뿐이면 진짜 특성인지 우연인지 검증 불가 -> 최소 2개는 있어야 신뢰 가능 -> **조합 전체 삭제**.

In [ ]:
only1_valid_codes = group_na[group_na == 2].index.tolist()
print(f"2회 결측(1회만 유효)인 조합 개수: {len(only1_valid_codes)}개")

## 5. 3단계 처리 - 같은 조합 3회 중 1회만 결측(2회는 유효)인 경우

무조건 삭제하지 않고, 남은 2개 값의 차이(range)를 확인 후 처리 방식을 나눈다.

### 5-1. 남은 2개 값의 차이(range) 분포 확인 -> 기준값 근거 마련

In [ ]:
ranges = []
for code_val, group_idx in df.groupby('Peltier_Level(6digits)').groups.items():
    sub = df.loc[group_idx]
    for col in sensor_cols:
        vals = sub[col]
        if vals.isnull().sum() == 1 and vals.notnull().sum() == 2:
            valid_vals = vals.dropna()
            rng = valid_vals.max() - valid_vals.min()
            ranges.append({'code': code_val, 'col': col, 'range': rng})

ranges_df = pd.DataFrame(ranges)
print("결측 1개(=2개 유효값) 케이스 총", len(ranges_df), "건")
print()
print(ranges_df['range'].describe())
print()
for q in [0.5, 0.75, 0.9, 0.95, 0.97, 0.98, 0.99, 1.0]:
    print(f"  {q*100:.0f}%: {ranges_df['range'].quantile(q):.3f}")

99% 데이터가 1.11도 이내, 그다음 값이 1.47도, 이상치가 12도 이상으로 압도적으로 튐 -> **1.5도**를 경계로 설정.

### 5-2. 1.5도 기준으로 평균 보완 적용

In [ ]:
RANGE_THRESHOLD = 1.5

imputed_log = []
skipped_outlier_log = []

for code_val, group_idx in df.groupby('Peltier_Level(6digits)').groups.items():
    sub = df.loc[group_idx]
    for col in sensor_cols:
        vals = sub[col]
        if vals.isnull().sum() == 1 and vals.notnull().sum() == 2:
            valid_vals = vals.dropna()
            rng = valid_vals.max() - valid_vals.min()
            if rng <= RANGE_THRESHOLD:
                mean_val = round(valid_vals.mean(), 2)
                missing_idx = vals[vals.isnull()].index
                df.loc[missing_idx, col] = mean_val
                imputed_log.append((code_val, col, rng, mean_val))
            else:
                skipped_outlier_log.append((code_val, col, rng))

print(f"평균으로 보완한 결측치: {len(imputed_log)}건")
print(f"이상치로 판단되어 보완하지 않은 경우: {len(skipped_outlier_log)}건")
print(skipped_outlier_log)

## 6. 최종 처리 - 남은 결측치 행 삭제

In [ ]:
before_rows = len(df)
before_combos = df['Peltier_Level(6digits)'].nunique()

df_final = df.dropna(subset=sensor_cols).drop(columns=['has_na']).reset_index(drop=True)

after_rows = len(df_final)
after_combos = df_final['Peltier_Level(6digits)'].nunique()

print("===== 최종 결측치 처리 요약 =====")
print(f"원본: {before_rows}행, {before_combos}개 조합")
print(f"평균 보완: {len(imputed_log)}건 (1.5도 이내 편차)")
print(f"행 삭제: {before_rows - after_rows}행 ({(before_rows-after_rows)/before_rows*100:.2f}%)")
print(f"최종: {after_rows}행, {after_combos}개 조합 ({after_combos/4096*100:.2f}% 커버)")
print()
print("최종 결측치 확인:")
print(df_final[sensor_cols].isnull().sum())

In [ ]:
df_final.head(10)

## 7. 이상치 처리

**주의: 첫 시도(자기 자신을 뺀 나머지 2개 평균과 비교하는 방식)는 구조적 결함이 있었음**

이상치가 하나만 있어도, 그 이상치가 "나머지 2개"의 평균 계산에 섞여 들어가서
정상값 2개까지 억울하게 이상치로 오탐되는 문제가 있었다.
(예: 23.83, 15.85, 24.00 -> 15.85만 이상치인데 3개 다 이상치로 잘못 판정됨)

**수정된 방식**: 3개 값을 정렬해서 양쪽 간격(gap)을 비교.
2개는 서로 가깝고(close_gap 작음) 1개만 고립된(isolation_gap 큼) 경우에만
그 고립값 하나만 이상치로 판정한다.

### 7-1. 간격(gap) 기반 이상치 탐지

In [ ]:
ISOLATION_THRESHOLD = 3.0   # 고립값과의 거리 기준
CLOSE_THRESHOLD = 1.5       # 정상 짝끼리는 이 이내로 가까워야 함

outlier_records = []

for code_val, group_idx in df_final.groupby('Peltier_Level(6digits)').groups.items():
    sub = df_final.loc[group_idx]
    if len(sub) != 3:
        continue
    for col in sensor_cols:
        idxs = np.array(sub.index.tolist())
        vals = sub[col].values
        order = np.argsort(vals)
        sorted_vals = vals[order]
        sorted_idxs = idxs[order]

        gap_low = sorted_vals[1] - sorted_vals[0]
        gap_high = sorted_vals[2] - sorted_vals[1]

        if gap_low > gap_high and gap_low > ISOLATION_THRESHOLD and gap_high <= CLOSE_THRESHOLD:
            outlier_records.append({
                'row_idx': sorted_idxs[0], 'code': code_val, 'col': col,
                'value': sorted_vals[0], 'isolation_gap': round(gap_low, 2)
            })
        elif gap_high > gap_low and gap_high > ISOLATION_THRESHOLD and gap_low <= CLOSE_THRESHOLD:
            outlier_records.append({
                'row_idx': sorted_idxs[2], 'code': code_val, 'col': col,
                'value': sorted_vals[2], 'isolation_gap': round(gap_high, 2)
            })

outlier_df = pd.DataFrame(outlier_records)
print(f"이상치로 판단된 셀: {len(outlier_records)}건")
print(f"영향받는 조합 수: {outlier_df['code'].nunique()}개")
print()
print("컬럼별 이상치 발생 건수:")
print(outlier_df['col'].value_counts())

### 7-2. 이상치를 결측치로 변환 후 평균 보완

In [ ]:
df_outlier_removed = df_final.copy()

for record in outlier_records:
    df_outlier_removed.loc[record['row_idx'], record['col']] = np.nan

print("이상치를 결측치로 변환 완료")
print(df_outlier_removed[sensor_cols].isnull().sum())

In [ ]:
IMPUTE_THRESHOLD = 1.5
imputed_count = 0

for code_val, group_idx in df_outlier_removed.groupby('Peltier_Level(6digits)').groups.items():
    sub = df_outlier_removed.loc[group_idx]
    for col in sensor_cols:
        vals = sub[col]
        if vals.isnull().sum() == 1 and vals.notnull().sum() == 2:
            valid_vals = vals.dropna()
            rng = valid_vals.max() - valid_vals.min()
            if rng <= IMPUTE_THRESHOLD:
                mean_val = round(valid_vals.mean(), 2)
                missing_idx = vals[vals.isnull()].index
                df_outlier_removed.loc[missing_idx, col] = mean_val
                imputed_count += 1

print(f"이상치 제거 후, 평균으로 보완한 셀: {imputed_count}건")
print(df_outlier_removed[sensor_cols].isnull().sum())

### 7-3. 최종 정리 및 저장

In [ ]:
before_rows2 = len(df_outlier_removed)
before_combos2 = df_outlier_removed['Peltier_Level(6digits)'].nunique()

df_clean = df_outlier_removed.dropna(subset=sensor_cols).reset_index(drop=True)

after_rows2 = len(df_clean)
after_combos2 = df_clean['Peltier_Level(6digits)'].nunique()

print("===== 이상치 처리 최종 요약 =====")
print(f"처리 전: {before_rows2}행, {before_combos2}개 조합")
print(f"이상치 탐지: {len(outlier_records)}건")
print(f"평균 보완: {imputed_count}건")
print(f"행 손실: {before_rows2 - after_rows2}행")
print(f"최종 데이터: {after_rows2}행, {after_combos2}개 조합 ({after_combos2/4096*100:.2f}% 커버)")

In [ ]:
SAVE_PATH_2 = r"C:\Users\hwan9\OneDrive\Desktop\프로젝트\스마트 팩토리\온도센서_csv\온도센서측정_이상치처리완료.csv"
df_clean.to_csv(SAVE_PATH_2, index=False, encoding='utf-8-sig')
print("저장 완료:", SAVE_PATH_2)

## 8. 파생변수 생성

- Ch1~Ch6: 제어값 6자리를 채널별로 분해
- Digit_Sum: 전체 세기 합 (6채널 전부 가열 방향이므로 = 총 가열량)
- Delta_T: 최종 온도변화량 (Sec_7 - Ref_Temp)

(Heat_Sum/Cool_Sum/Heat_Cool_Diff는 6채널이 전부 가열 방향임을 확인하고 제외함)

In [ ]:
for i in range(6):
    df_clean[f'Ch{i+1}'] = df_clean['Peltier_Level(6digits)'].str[i].astype(int)

df_clean['Digit_Sum'] = df_clean[[f'Ch{i+1}' for i in range(6)]].sum(axis=1)
df_clean['Delta_T'] = df_clean['Sec_7'] - df_clean['Ref_Temp']

print("파생변수 생성 완료")
df_clean[['Peltier_Level(6digits)', 'Ch1','Ch2','Ch3','Ch4','Ch5','Ch6',
          'Digit_Sum', 'Delta_T']].head(10)

In [ ]:
print(df_clean[['Digit_Sum', 'Delta_T']].describe())

In [ ]:
plt.figure(figsize=(8,6))
plt.scatter(df_clean['Digit_Sum'], df_clean['Delta_T'], alpha=0.1, s=5)
plt.xlabel('Digit_Sum (총 세기 합)')
plt.ylabel('Delta_T (Sec_7 - Ref_Temp)')
plt.title('총 세기와 온도변화의 관계')
plt.grid(True, alpha=0.3)
plt.show()

print("상관계수:", df_clean['Digit_Sum'].corr(df_clean['Delta_T']))

In [ ]:
SAVE_PATH_3 = r"C:\Users\hwan9\OneDrive\Desktop\프로젝트\스마트 팩토리\온도센서_csv\온도센서측정_파생변수완료.csv"
df_clean.to_csv(SAVE_PATH_3, index=False, encoding='utf-8-sig')
print("저장 완료:", SAVE_PATH_3)

## 9. 채널별 영향력 확인

Digit_Sum은 6채널을 모두 동일하게 취급하는데, 실제로 채널마다 
Sec_7과의 물리적 거리가 달라 영향력이 다를 수 있다는 가설을 검증한다.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

channel_cols = ['Ch1', 'Ch2', 'Ch3', 'Ch4', 'Ch5', 'Ch6']

for i, ch in enumerate(channel_cols):
    other_channels = [c for c in channel_cols if c != ch]
    mask = (df_clean[other_channels] == 0).all(axis=1)
    sub = df_clean[mask]
    grouped = sub.groupby(ch)['Delta_T'].agg(['mean', 'std', 'count'])
    axes[i].errorbar(grouped.index, grouped['mean'], yerr=grouped['std'], marker='o', capsize=5)
    axes[i].set_title(f'{ch} 단독 영향력')
    axes[i].set_xlabel(f'{ch} 세기(0~3)')
    axes[i].set_ylabel('Delta_T 평균')
    axes[i].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
print("===== 각 채널 단독 3단계일 때 평균 Delta_T =====")
for ch in channel_cols:
    other_channels = [c for c in channel_cols if c != ch]
    mask = (df_clean[other_channels] == 0).all(axis=1) & (df_clean[ch] == 3)
    sub = df_clean[mask]
    if len(sub) > 0:
        print(f"{ch}=3 (나머지 0): 평균 Delta_T = {sub['Delta_T'].mean():.2f}도, 표본수 = {len(sub)}개")

print()
print("===== 채널별 Delta_T와의 상관계수 =====")
for ch in channel_cols:
    corr = df_clean[ch].corr(df_clean['Delta_T'])
    print(f"{ch}: {corr:.4f}")

**결론**: 채널마다 약간의 차이는 있으나(Ch1이 가장 약하고 Ch5/Ch6이 가장 강함, 
최대 약 0.6도 차이), 가중치를 반영한 Weighted_Sum과 단순 Digit_Sum의 상관계수 차이가
0.0016에 불과해(0.9449 vs 0.9465) 실용적으로는 단순 Digit_Sum을 그대로 채택한다.

## 10. 불량 판정 기준 수립

Digit_Sum으로 Delta_T를 예측하는 회귀식을 구하고, 실제값과 예측값의 
차이(잔차, Residual)가 허용범위를 벗어나면 불량으로 판정한다.

In [ ]:
from sklearn.linear_model import LinearRegression

X_reg = df_clean[['Digit_Sum']].values
y_reg = df_clean['Delta_T'].values

reg = LinearRegression()
reg.fit(X_reg, y_reg)

slope = reg.coef_[0]
intercept = reg.intercept_

print(f"회귀식: Delta_T = {slope:.4f} x Digit_Sum + {intercept:.4f}")
print(f"R² (설명력): {reg.score(X_reg, y_reg):.4f}")

In [ ]:
df_clean['Expected_Delta_T'] = slope * df_clean['Digit_Sum'] + intercept
df_clean['Residual'] = df_clean['Delta_T'] - df_clean['Expected_Delta_T']

print(df_clean[['Peltier_Level(6digits)', 'Digit_Sum', 'Delta_T', 'Expected_Delta_T', 'Residual']].head(10))
print()
print("잔차 기초 통계:")
print(df_clean['Residual'].describe())

In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(df_clean['Residual'], bins=100)
plt.xlabel('Residual (실제 - 기대값)')
plt.ylabel('개수')
plt.title('잔차 분포')
plt.axvline(0, color='red', linestyle='--')
plt.show()

print("분위수:")
for q in [0.01, 0.05, 0.1, 0.9, 0.95, 0.99]:
    print(f"  {q*100:.0f}%: {df_clean['Residual'].quantile(q):.3f}")
print()
print("표준편차:", df_clean['Residual'].std())

**허용오차 ±2.0도로 설정** (95% 분위수 1.944도를 살짝 넘는 지점, 
전체의 약 90~95%가 정상 범위에 들어오도록 설계)

In [ ]:
TOLERANCE = 2.0

df_clean['Label'] = (df_clean['Residual'].abs() <= TOLERANCE).astype(int)

print(df_clean['Label'].value_counts())
print()
print(f"양품 비율: {(df_clean['Label']==1).mean()*100:.2f}%")
print(f"불량 비율: {(df_clean['Label']==0).mean()*100:.2f}%")

### 10-1. 불량 케이스 분석

In [ ]:
defect_df = df_clean[df_clean['Label'] == 0]

print(f"불량 건수: {len(defect_df)}건")
print()
print("불량 케이스의 Digit_Sum 분포:")
print(defect_df['Digit_Sum'].describe())
print()

df_clean['Digit_Sum_bin'] = pd.cut(df_clean['Digit_Sum'], bins=[0,3,6,9,12,15,18], include_lowest=True)
defect_rate_by_bin = df_clean.groupby('Digit_Sum_bin', observed=True)['Label'].apply(lambda x: (x==0).mean()*100)
print("Digit_Sum 구간별 불량 비율(%):")
print(defect_rate_by_bin)

**발견**: 세기가 셀수록 불량이 많을 것이라는 예상과 달리, 중간 세기(6~9) 구간에서 
불량 비율이 가장 높고(12.87%), 최대 세기(15~18) 구간은 오히려 0%로 나타났다.
절대적인 세기보다 상대적인 변화 비율이 판정에 영향을 미쳤을 가능성을 시사한다.

In [ ]:
plt.figure(figsize=(10, 7))
plt.scatter(df_clean[df_clean['Label']==1]['Digit_Sum'], df_clean[df_clean['Label']==1]['Delta_T'],
            alpha=0.2, s=8, color='steelblue', label='양품')
plt.scatter(df_clean[df_clean['Label']==0]['Digit_Sum'], df_clean[df_clean['Label']==0]['Delta_T'],
            alpha=0.5, s=8, color='red', label='불량')
x_line = sorted(df_clean['Digit_Sum'].unique())
plt.plot(x_line, [slope*x+intercept for x in x_line], color='black', linewidth=1, label='기대값 회귀선')
plt.xlabel('Digit_Sum')
plt.ylabel('Delta_T')
plt.title('양품/불량 분포')
plt.legend()
plt.show()

In [ ]:
SAVE_PATH_4 = r"C:\Users\hwan9\OneDrive\Desktop\프로젝트\스마트 팩토리\온도센서_csv\온도센서측정_라벨링완료.csv"
df_clean.to_csv(SAVE_PATH_4, index=False, encoding='utf-8-sig')
print("저장 완료:", SAVE_PATH_4)

## 11. 머신러닝 모델링

원본 센서값(제어값 + 온도값)만으로 ML이 Label(양품/불량) 패턴을 
스스로 재발견할 수 있는지 확인한다.

In [ ]:
from sklearn.model_selection import train_test_split

feature_cols = ['Ch1','Ch2','Ch3','Ch4','Ch5','Ch6',
                 'Ref_Temp','Sec_2','Sec_3','Sec_4','Sec_5','Sec_6','Sec_7','Delta_T']

X = df_clean[feature_cols]
y = df_clean['Label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {len(X_train)}개 / Test: {len(X_test)}개")
print(f"Train 양품비율: {(y_train==1).mean()*100:.1f}%")
print(f"Test 양품비율: {(y_test==1).mean()*100:.1f}%")

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

models = {
    '로지스틱회귀': LogisticRegression(max_iter=1000),
    '결정트리(depth=10)': DecisionTreeClassifier(max_depth=10, random_state=42),
    '랜덤포레스트': RandomForestClassifier(n_estimators=200, max_depth=15,
                                          class_weight='balanced', random_state=42),
    'GradientBoosting': GradientBoostingClassifier(n_estimators=200, max_depth=5, random_state=42)
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    acc = accuracy_score(y_test, pred)
    cm = confusion_matrix(y_test, pred)
    results[name] = {'model': model, 'pred': pred, 'accuracy': acc, 'cm': cm}
    print(f"\n===== {name} =====")
    print(f"Accuracy: {acc:.4f}")
    print("Confusion Matrix:")
    print(cm)
    print(classification_report(y_test, pred, target_names=['불량','양품']))

In [ ]:
importances = results['랜덤포레스트']['model'].feature_importances_
importance_df = pd.DataFrame({'feature': feature_cols, 'importance': importances}).sort_values('importance', ascending=False)
print(importance_df)

plt.figure(figsize=(10,6))
plt.barh(importance_df['feature'], importance_df['importance'])
plt.xlabel('중요도')
plt.title('랜덤포레스트 변수 중요도')
plt.gca().invert_yaxis()
plt.show()

## 12. 다음 단계

여기까지 결측치/이상치 처리, 파생변수 생성, 채널별 영향력 분석, 
불량 판정 기준 수립, 머신러닝 모델링을 완료했다. 
결과를 바탕으로 추가 분석이나 보고서 작성을 이어서 진행한다.